# Classificazione con scikit-learn e statsmodels: quattro algoritmi, quattro dataset

In questo notebook applichiamo quattro algoritmi di classificazione ad altrettanti dataset sintetici,
costruiti sui dati che un'azienda software produce ogni giorno: account dei clienti, esecuzioni dei
test, ticket di supporto, pull request. Come nel notebook sulla regressione, ogni algoritmo viene
applicato a **un solo** dataset, scelto perché ne mette in luce i punti di forza e i limiti.

| Sezione | Dataset | Target | Classi | Algoritmo | Perché proprio qui |
|---|---|---|---|---|---|
| 3 | `churn_saas.csv` | `churn` | binario, ~7% positivi | **Regressione logistica** (scikit-learn + **statsmodels** per l'inferenza) | i coefficienti si leggono come *odds ratio* ("ogni ritardo di pagamento moltiplica il rischio per..."), e le probabilità calibrate permettono di scegliere la soglia in base ai costi |
| 4 | `test_flaky.csv` | `flaky` | binario, ~46% positivi | **Albero decisionale** | il team riconosce un test flaky con poche regole ("fallisce da solo e il modulo non è stato toccato"): l'albero le trova e le mostra |
| 5 | `triage_ticket.csv` | `categoria` | 4 classi, sbilanciate | **k-Nearest Neighbors** | "a quali ticket già classificati assomiglia questo?" è la logica di chi fa triage a mano; il multiclasse viene gratis |
| 6 | `defect_pr.csv` | `bug_entro_30gg` | binario, ~12% positivi | **Rete neurale MLP** | il rischio nasce da *combinazioni* di metriche (modifica grande **senza** test, codice complesso **e** poco coperto): la rete le impara, un modello additivo no |

**Come misuriamo la qualità.** Per ogni modello riportiamo **accuracy, precision, recall, F1 e
ROC-AUC** su dati mai visti in addestramento, sempre accanto alla **matrice di confusione**, che è il
vero oggetto da leggere. Come punto di riferimento usiamo il classificatore ingenuo che predice
sempre la classe più frequente: su un dataset con il 7% di churn ha un'accuracy del 93% senza aver
imparato nulla, ed è esattamente il motivo per cui l'accuracy da sola non basta.

Anche qui due strumenti complementari: un **test set separato** (`train_test_split` stratificato,
così le classi rare sono rappresentate allo stesso modo nei due insiemi) che tocchiamo una sola volta,
e la **cross-validazione stratificata a 5 fold** sul solo training per scegliere gli iperparametri.

**Struttura.** Analisi esplorativa di ogni dataset (bilanciamento delle classi, distribuzione delle
feature per classe, correlazioni); scelte comuni di preparazione; i quattro modelli, ottimizzati e
valutati; riepilogo finale.

**Requisiti ed esecuzione in VS Code.** Servono `pandas`, `numpy`, `matplotlib`, `seaborn`,
`scikit-learn` (≥ 1.2) e `statsmodels`; in un ambiente conda:

```
conda install pandas numpy matplotlib seaborn scikit-learn statsmodels ipykernel
```

Apriamo il notebook in VS Code, scegliamo il kernel dell'ambiente (*Select Kernel*) e usiamo
*Run All*: l'esecuzione completa richiede un paio di minuti (la ricerca degli iperparametri della
rete neurale è la parte più lenta). I quattro CSV devono trovarsi nella stessa cartella del notebook,
oppure modifichiamo `DATA_DIR` nella cella di setup.

## 0. Setup: librerie, configurazione e funzioni di supporto

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.calibration import CalibrationDisplay

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4.5)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RANDOM_STATE = 42              # seme unico: risultati riproducibili in ogni cella
DATA_DIR = Path(".")           # cartella che contiene i quattro CSV

`statsmodels` lo importiamo a parte: lo usiamo soltanto nella sezione 3, per l'**inferenza** sulla
regressione logistica (errori standard, p-value, intervalli di confidenza degli odds ratio, effetti
marginali), cioè per tutto ciò che scikit-learn, orientato alla previsione, non fornisce.

In [ ]:
import statsmodels.formula.api as smf

Le funzioni di supporto: calcolo delle metriche (binarie e multiclasse), matrice di confusione,
riepilogo di un dataset con il bilanciamento delle classi, e i grafici esplorativi che ripeteremo
su ogni dataset.

In [ ]:
def metriche_binarie(y_vero, y_pred, proba=None, etichetta=""):
    '''Accuracy, precision, recall e F1 sulla classe positiva; ROC-AUC se abbiamo le probabilità.'''
    m = {"modello": etichetta,
         "accuracy": accuracy_score(y_vero, y_pred),
         "precision": precision_score(y_vero, y_pred, zero_division=0),
         "recall": recall_score(y_vero, y_pred),
         "F1": f1_score(y_vero, y_pred)}
    if proba is not None:
        m["ROC_AUC"] = roc_auc_score(y_vero, proba)
    return m


def metriche_multiclasse(y_vero, y_pred, proba=None, classi=None, etichetta=""):
    '''Come sopra ma con medie "macro": ogni classe pesa allo stesso modo, anche se rara.'''
    m = {"modello": etichetta,
         "accuracy": accuracy_score(y_vero, y_pred),
         "precision": precision_score(y_vero, y_pred, average="macro", zero_division=0),
         "recall": recall_score(y_vero, y_pred, average="macro"),
         "F1": f1_score(y_vero, y_pred, average="macro")}
    if proba is not None:
        m["ROC_AUC"] = roc_auc_score(y_vero, proba, multi_class="ovr", average="macro", labels=classi)
    return m


def stampa_metriche(m):
    '''Stampa le metriche numeriche di un dizionario su una riga.'''
    print("   ".join(f"{k} = {v:.3f}" for k, v in m.items() if isinstance(v, (int, float, np.floating))))


def mostra_confusione(y_vero, y_pred, etichette, titolo, ax=None, normalizza=None):
    '''Matrice di confusione: righe = classe vera, colonne = classe prevista.'''
    cm = confusion_matrix(y_vero, y_pred, labels=etichette, normalize=normalizza)
    disp = ConfusionMatrixDisplay(cm, display_labels=etichette)
    disp.plot(ax=ax, cmap="Blues", values_format=".2f" if normalizza else "d", colorbar=False)
    disp.ax_.set_title(titolo)
    disp.ax_.set_xlabel("classe prevista")
    disp.ax_.set_ylabel("classe vera")


def riepilogo(df, nome, target):
    '''Dimensioni, tipi, valori mancanti e bilanciamento delle classi; restituisce le statistiche descrittive.'''
    print(f"=== {nome}: {df.shape[0]} righe x {df.shape[1]} colonne ===\n")
    print("Tipi di dato:")
    print(df.dtypes.to_string(), "\n")
    print("Valori mancanti totali:", int(df.isna().sum().sum()), "\n")
    distribuzione = pd.DataFrame({"conteggio": df[target].value_counts(),
                                  "quota": df[target].value_counts(normalize=True).round(3)})
    print(f"Distribuzione del target '{target}':")
    print(distribuzione.to_string(), "\n")
    print("Statistiche descrittive (numeriche e categoriche):")
    return df.describe(include="all").T


def heatmap_correlazioni(df, titolo):
    '''Matrice di correlazione di Pearson tra le colonne numeriche (target 0/1 compreso, se numerico).'''
    corr = df.select_dtypes("number").corr()
    fig, ax = plt.subplots(figsize=(1.0 * len(corr) + 2, 0.7 * len(corr) + 1.5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax, annot_kws={"size": 8})
    ax.set_title(titolo)
    plt.tight_layout()
    plt.show()


def boxplot_per_classe(df, colonne, target, scala_log=(), n_col=4):
    '''Una griglia di boxplot: distribuzione di ogni feature numerica nelle diverse classi.'''
    n_righe = int(np.ceil(len(colonne) / n_col))
    fig, axes = plt.subplots(n_righe, n_col, figsize=(4.2 * n_col, 3.6 * n_righe))
    for ax, col in zip(np.ravel(axes), colonne):
        sns.boxplot(data=df, x=target, y=col, ax=ax)
        if col in scala_log:
            ax.set_yscale("log")
        ax.set_title(col)
        ax.set_xlabel("")
    for ax in np.ravel(axes)[len(colonne):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def barre_categoria_per_classe(df, colonne, target, n_col=3):
    '''Per ogni feature categorica: composizione delle classi del target in ciascuna categoria.'''
    n_righe = int(np.ceil(len(colonne) / n_col))
    fig, axes = plt.subplots(n_righe, n_col, figsize=(5.2 * n_col, 3.8 * n_righe))
    for ax, col in zip(np.ravel(axes), colonne):
        pd.crosstab(df[col], df[target], normalize="index").plot.bar(stacked=True, ax=ax, rot=0)
        ax.set_title(f"{target} per {col}")
        ax.set_ylabel("quota")
        ax.legend(title=target, fontsize=8)
    for ax in np.ravel(axes)[len(colonne):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

Carichiamo i quattro dataset e creiamo subito, per il churn, la feature `tasso_utilizzo` (utenti
attivi diviso licenze): 40 utenti attivi su 50 licenze sono un ottimo segno, 40 su 300 un allarme.
È la grandezza che conta, e da sola nessuna delle due colonne originali la esprime.

In [ ]:
churn = pd.read_csv(DATA_DIR / "churn_saas.csv")
flaky = pd.read_csv(DATA_DIR / "test_flaky.csv")
triage = pd.read_csv(DATA_DIR / "triage_ticket.csv")
defect = pd.read_csv(DATA_DIR / "defect_pr.csv")

churn["tasso_utilizzo"] = (churn["utenti_attivi_30gg"] / churn["licenze"]).round(3)

for nome, df in {"churn_saas": churn, "test_flaky": flaky, "triage_ticket": triage, "defect_pr": defect}.items():
    print(f"{nome:16s} {df.shape[0]:5d} righe x {df.shape[1]:2d} colonne")

## 1. Analisi esplorativa dei dati

Per ogni dataset rispondiamo alle stesse domande, e le risposte guidano le scelte di modellazione:

1. **Quanti dati abbiamo e di che tipo?** Dimensioni, tipi, valori mancanti.
2. **Quanto sono sbilanciate le classi?** È la domanda più importante in classificazione: decide
   quali metriche hanno senso, se serve stratificare, se la soglia di decisione va scelta con cura.
3. **Quali feature separano le classi?** Boxplot delle numeriche per classe, composizione delle
   classi per ogni categoria delle categoriche, correlazioni (per i target binari possiamo includere
   il target stesso, codificato 0/1, nella matrice).
4. **Cosa ci suggerisce sul modello?** Trasformazioni, codifiche, feature da escludere, metriche.

### 1.1 `churn_saas`: quali clienti non rinnoveranno

2.000 account di un prodotto SaaS: piano e tipo di fatturazione, anzianità, licenze e utilizzo,
intensità d'uso, ticket e tempi di risposta del supporto, ritardi di pagamento. Il target `churn`
vale 1 se l'account non ha rinnovato nei 90 giorni successivi.

In [ ]:
riepilogo(churn, "churn_saas", "churn")

In [ ]:
boxplot_per_classe(churn, ["tasso_utilizzo", "login_medi_settimana", "feature_usate", "integrazioni_attive",
                           "ticket_90gg", "tempo_risposta_medio_ore", "ritardi_pagamento_12m", "mesi_anzianita"],
                   "churn", scala_log=("tempo_risposta_medio_ore",))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, col in zip(axes[:2], ["piano", "fatturazione"]):
    churn.groupby(col)["churn"].mean().plot.bar(ax=ax, color="steelblue", rot=0)
    ax.set_title(f"Tasso di churn per {col}")
    ax.set_ylabel("quota di churn")
churn.groupby("ritardi_pagamento_12m")["churn"].mean().plot.bar(ax=axes[2], color="indianred", rot=0)
axes[2].set_title("Tasso di churn per numero di ritardi di pagamento")
axes[2].set_ylabel("quota di churn")
plt.tight_layout()
plt.show()

heatmap_correlazioni(churn.drop(columns=["id_cliente"]), "Correlazioni: churn_saas (target compreso)")

**Cosa osserviamo.**

- Le classi sono **molto sbilanciate** (circa 7% di churn): l'accuracy del classificatore ingenuo è
  già ~93%. Le metriche da guardare sono recall (quanti clienti a rischio intercettiamo), precision
  (quante segnalazioni sono giuste), ROC-AUC e la matrice di confusione.
- I segnali più netti sono il **tasso di utilizzo** delle licenze, i **ritardi di pagamento** (il churn
  cresce quasi linearmente con il loro numero) e la **fatturazione mensile**; contano anche feature
  usate, integrazioni e anzianità. Nessuna variabile da sola separa le classi: gli effetti si sommano,
  ed è proprio la situazione in cui un modello lineare sui log-odds funziona bene.
- Le correlazioni con il target sono tutte modeste in valore assoluto, com'è normale con un target
  binario raro: la matrice serve più a vedere le relazioni tra feature (`licenze` è legata al piano).

### 1.2 `test_flaky`: fallimento intermittente o bug reale?

1.200 fallimenti di test in CI. Per ognuno conosciamo lo storico del test (tasso di fallimento,
alternanze pass/fail negli ultimi 20 run, fallimenti consecutivi), il suo comportamento (varia di
durata? usa rete, database, sleep o timeout? fallisce solo su alcuni runner?) e il contesto del
commit (altri test falliti, il modulo testato è stato modificato?). Il target `flaky` vale 1 se il
fallimento si è poi rivelato instabile, 0 se segnalava un bug vero.

In [ ]:
riepilogo(flaky, "test_flaky", "flaky")

In [ ]:
boxplot_per_classe(flaky, ["tasso_fallimento_storico", "alternanze_ultimi_20", "fallimenti_consecutivi",
                           "cv_durata", "altri_test_falliti_stesso_commit", "durata_media_s"],
                   "flaky", scala_log=("durata_media_s",), n_col=3)

flag = ["solo_su_alcuni_runner", "usa_rete", "usa_db", "usa_sleep_timeout", "modulo_modificato_nel_commit"]
fig, ax = plt.subplots(figsize=(11, 4))
flaky.groupby("flaky")[flag].mean().T.plot.bar(ax=ax, rot=15)
ax.set_title("Quota di fallimenti con ciascun flag attivo, per classe (0 = rotto, 1 = flaky)")
ax.set_ylabel("quota")
plt.tight_layout()
plt.show()

heatmap_correlazioni(flaky.drop(columns=["id_fallimento"]), "Correlazioni: test_flaky (target compreso)")

**Cosa osserviamo.**

- Le classi sono quasi **bilanciate** (46% flaky): qui l'accuracy è una metrica sensata, e il
  classificatore ingenuo si ferma al 54%.
- Alcune feature separano le classi in modo netto: i flaky hanno molte **alternanze** pass/fail e
  uno storico di fallimenti intermittenti; i test rotti falliscono **consecutivamente** dopo un commit
  che ha **modificato il modulo** e tirano giù **altri test** dello stesso commit.
- I flag di comportamento (rete, sleep/timeout, runner specifici, durata variabile) alzano la
  probabilità di flaky ma non sono decisivi da soli.
- La struttura "se... allora..." di queste evidenze è esattamente quella che un albero decisionale
  rappresenta: ci aspettiamo poche regole con alta accuratezza. `durata_media_s` sembra irrilevante:
  vedremo se l'albero la ignora.

### 1.3 `triage_ticket`: a chi va questo ticket?

2.000 ticket con la categoria assegnata a mano dal supporto (`Bug`, `Richiesta` di funzionalità,
`Domanda`, `Incidente`). Le feature sono i metadati disponibili all'apertura: canale, tipo di
cliente, componente e priorità indicati dall'utente, presenza di stack trace, allegati, e alcuni
conteggi ricavati dal testo (parole legate a errori, parole che esprimono una richiesta, punti
interrogativi, lunghezza), più il numero di utenti che segnalano lo stesso problema.

In [ ]:
riepilogo(triage, "triage_ticket", "categoria")

In [ ]:
boxplot_per_classe(triage, ["parole_errore", "parole_richiesta", "punti_interrogativi",
                            "utenti_segnalanti", "lunghezza_descrizione", "allegati"],
                   "categoria", scala_log=("utenti_segnalanti", "lunghezza_descrizione"), n_col=3)

barre_categoria_per_classe(triage, ["canale", "priorita_dichiarata", "componente_indicato"], "categoria")

fig, ax = plt.subplots(figsize=(10, 3.8))
triage.groupby("categoria")["stack_trace"].mean().plot.bar(ax=ax, color="steelblue", rot=0)
ax.set_title("Quota di ticket con stack trace, per categoria")
plt.tight_layout()
plt.show()

**Cosa osserviamo.**

- Quattro classi con frequenze diverse: `Bug` è il 40%, `Incidente` solo il 10%. Con la media
  "macro" delle metriche ogni classe pesa allo stesso modo, così un modello che ignora gli incidenti
  (la classe più costosa da sbagliare) viene penalizzato.
- Ogni classe ha la sua firma: gli **incidenti** arrivano per telefono o chat, con priorità alta e
  **molti utenti che segnalano** lo stesso problema; i **bug** portano stack trace, allegati e parole
  di errore; le **richieste** usano il lessico del "vorrei/sarebbe possibile"; le **domande** sono
  brevi, piene di punti interrogativi, spesso su fatturazione o account.
- Le classi si somigliano a coppie: Bug e Incidente condividono il lessico degli errori, Richiesta e
  Domanda il tono e i canali. Vedremo dalla matrice di confusione quale coppia il modello distingue
  peggio.
- Le feature sono un misto di numeriche su scale diverse e categoriche: per il kNN servono scaling e
  one-hot, altrimenti la distanza sarebbe dominata da `lunghezza_descrizione`.

### 1.4 `defect_pr`: quali pull request porteranno un bug

4.000 pull request già unite nel ramo principale. Per ognuna abbiamo dimensione e forma della
modifica (righe, file, moduli, complessità), se ha aggiunto test, la copertura del modulo toccato,
esperienza dell'autore, review ricevuta, momento del merge e tipo di PR. Il target
`bug_entro_30gg` vale 1 se nei 30 giorni successivi è stato necessario un fix su quel codice.

In [ ]:
riepilogo(defect, "defect_pr", "bug_entro_30gg")

In [ ]:
defect["righe_totali"] = defect["righe_aggiunte"] + defect["righe_rimosse"]
boxplot_per_classe(defect, ["righe_totali", "complessita_media", "copertura_modulo", "esperienza_autore_mesi",
                            "moduli_toccati", "commenti_review", "reviewer", "commit_autore_modulo"],
                   "bug_entro_30gg", scala_log=("righe_totali",))

In [ ]:
# Il tasso di bug in funzione di COPPIE di condizioni: è qui che si nasconde la struttura del problema
defect["modifica_grande"] = (defect["righe_totali"] > 300).map({True: "> 300 righe", False: "≤ 300 righe"})
defect["copertura_bassa"] = (defect["copertura_modulo"] < 50).map({True: "copertura < 50%", False: "copertura ≥ 50%"})
defect["complessa"] = (defect["complessita_media"] > 10).map({True: "complessità > 10", False: "complessità ≤ 10"})

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
pd.crosstab(defect["modifica_grande"], defect["test_aggiunti"], values=defect["bug_entro_30gg"], aggfunc="mean") \
    .plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Tasso di bug: dimensione × test aggiunti")
axes[0].legend(title="test_aggiunti")
pd.crosstab(defect["copertura_bassa"], defect["complessa"], values=defect["bug_entro_30gg"], aggfunc="mean") \
    .plot.bar(ax=axes[1], rot=0)
axes[1].set_title("Tasso di bug: copertura × complessità")
defect.groupby("ora_merge")["bug_entro_30gg"].mean().plot.bar(ax=axes[2], color="steelblue")
axes[2].set_title("Tasso di bug per ora del merge")
for ax in axes:
    ax.set_ylabel("quota di bug")
plt.tight_layout()
plt.show()

heatmap_correlazioni(defect.drop(columns=["id_pr"]), "Correlazioni: defect_pr (target compreso)")

**Cosa osserviamo.**

- Classi **sbilanciate** (circa 12% di PR con bug): come per il churn, precision, recall, F1 e
  ROC-AUC contano più dell'accuracy; in più, la soglia di decisione andrà scelta.
- Prese una alla volta, le feature separano poco le classi (i boxplot si sovrappongono quasi tutti).
  Ma le **coppie** raccontano un'altra storia: una modifica grande *senza* test, o un modulo poco
  coperto *e* complesso, hanno tassi di bug molto più alti della somma dei singoli effetti. Il
  rischio nasce dalle **interazioni**.
- L'effetto dell'ora del merge non è monotono: merge notturni e serali sono più rischiosi.
- Un modello additivo (logistico) può cogliere solo una versione appiattita di tutto questo; una rete
  neurale, a patto di regolarizzarla bene, può rappresentare le congiunzioni. È il confronto che
  faremo nella sezione 6. Le colonne ausiliarie create qui (`modifica_grande`, ...) servono solo alla
  lettura: ai modelli daremo le variabili originali.

## 2. Preparazione comune: split stratificato, pipeline, metriche e soglie

**Split stratificato.** `train_test_split(..., stratify=y)` e `StratifiedKFold` mantengono in
ogni sottoinsieme le stesse proporzioni di classe del dataset. Con il 7% di positivi, uno split
casuale potrebbe lasciare nel test troppo pochi casi rari per valutare qualcosa.

**Pipeline.** Come nella regressione: trasformazioni stimate sul training e riapplicate al test,
dentro una `Pipeline`, per evitare *data leakage*. One-hot per le categoriche, standardizzazione per
kNN e rete neurale, nulla per l'albero.

**Le metriche, in breve.** Dalla matrice di confusione (VP = veri positivi, FP = falsi positivi,
FN = falsi negativi, VN = veri negativi):

- **accuracy** = (VP + VN) / totale: la quota di previsioni giuste. Ingannevole con classi sbilanciate.
- **precision** = VP / (VP + FP): quando il modello dice "positivo", quante volte ha ragione.
- **recall** = VP / (VP + FN): quanti positivi veri il modello riesce a intercettare.
- **F1**: media armonica di precision e recall, alta solo se lo sono entrambe.
- **ROC-AUC**: la probabilità che un positivo preso a caso riceva un punteggio più alto di un negativo
  preso a caso. Non dipende dalla soglia: misura la qualità dell'*ordinamento*, 0.5 = caso, 1 = perfetto.

**La soglia.** Un classificatore probabilistico produce una probabilità; la classe si ottiene
confrontandola con una soglia, per default 0.5. Con classi sbilanciate 0.5 è quasi sempre una
cattiva scelta: sposta tutto verso la classe maggioritaria. Spostare la soglia scambia precision con
recall, e la soglia giusta dipende dai **costi** dell'errore: lo vedremo sul churn (costo di un
cliente perso contro costo di una telefonata) e sulle PR.

**Metrica di selezione degli iperparametri.** In cross-validazione usiamo la metrica coerente con
ciascun problema: ROC-AUC per churn e PR (classi sbilanciate, ci interessa l'ordinamento), accuracy
per i test flaky (classi bilanciate), F1 macro per il triage (multiclasse con una classe rara).

In [ ]:
risultati = []                                                               # una riga per modello, per il riepilogo finale
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)    # le stesse 5 pieghe stratificate per tutti i modelli


def accuracy_ingenua(y_train, y_test):
    '''Accuracy del classificatore che predice sempre la classe più frequente del training.'''
    ingenuo = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_train), 1)), y_train)
    return accuracy_score(y_test, ingenuo.predict(np.zeros((len(y_test), 1))))

## 3. Regressione logistica: `churn_saas`

La regressione logistica modella il **logaritmo delle odds** (probabilità di churn diviso
probabilità di non churn) come combinazione lineare delle feature:

$$\log\frac{p}{1-p} = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k
\qquad\Longrightarrow\qquad p = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots)}}$$

Ogni coefficiente ha un'interpretazione diretta: $e^{\beta_j}$ è l'**odds ratio**, cioè il fattore
per cui le odds di churn vengono moltiplicate quando $x_j$ aumenta di un'unità, a parità delle altre
variabili. È il modello che un responsabile del customer success può leggere e discutere, ed è
naturalmente calibrato: la probabilità che produce *è* una probabilità, e possiamo usarla per
decidere chi chiamare.

Con scikit-learn costruiamo la pipeline e misuriamo; con statsmodels stimiamo lo stesso modello per
l'inferenza. Un dettaglio: scikit-learn applica di default una regolarizzazione L2 (`C=1`), che
restringe leggermente i coefficienti. La disattiviamo in pratica con `C` molto grande, così le due
librerie stimano esattamente lo stesso modello e possiamo verificarlo confrontando le previsioni.

In [ ]:
num_churn = ["tasso_utilizzo", "licenze", "mesi_anzianita", "login_medi_settimana", "feature_usate",
             "integrazioni_attive", "ticket_90gg", "tempo_risposta_medio_ore", "ritardi_pagamento_12m"]
cat_churn = ["piano", "fatturazione"]
X_churn, y_churn = churn[num_churn + cat_churn], churn["churn"]
X_train_churn, X_test_churn, y_train_churn, y_test_churn = train_test_split(
    X_churn, y_churn, test_size=0.2, stratify=y_churn, random_state=RANDOM_STATE)
print(f"training: {len(X_train_churn)} account ({y_train_churn.mean():.1%} churn)   "
      f"test: {len(X_test_churn)} account ({y_test_churn.mean():.1%} churn)")

prep_churn = ColumnTransformer([
    ("num", StandardScaler(), num_churn),
    ("cat", OneHotEncoder(drop="first"), cat_churn),      # una dummy in meno: la categoria di riferimento va nell'intercetta
], sparse_threshold=0, verbose_feature_names_out=False)

logistica_churn = Pipeline([("prep", prep_churn),
                            ("logit", LogisticRegression(C=1e6, max_iter=5000))])   # C grande = niente regolarizzazione

for metrica in ["roc_auc", "average_precision"]:
    punteggi = cross_val_score(logistica_churn, X_train_churn, y_train_churn, cv=cv, scoring=metrica)
    print(f"{metrica:18s} in cross-validazione: {punteggi.mean():.3f} ± {punteggi.std():.3f}")

logistica_churn.fit(X_train_churn, y_train_churn)
proba_churn = logistica_churn.predict_proba(X_test_churn)[:, 1]      # probabilità di churn stimata
pred_churn_05 = (proba_churn >= 0.5).astype(int)                     # decisione con la soglia di default

Guardiamo il risultato con la soglia di default (0.5), poi le tre curve che descrivono il modello
indipendentemente dalla soglia: la **curva ROC** (recall contro tasso di falsi positivi), la curva
**precision-recall**, più informativa della ROC quando i positivi sono rari, e la **curva di
calibrazione** (quando il modello dice "30% di rischio", i clienti churnano davvero nel 30% dei casi?).

In [ ]:
m05 = metriche_binarie(y_test_churn, pred_churn_05, proba_churn, "Regressione logistica (soglia 0.5)")
stampa_metriche(m05)
print(f"accuracy del classificatore ingenuo (sempre 'non churn'): {accuracy_ingenua(y_train_churn, y_test_churn):.3f}")

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
mostra_confusione(y_test_churn, pred_churn_05, [0, 1], "Matrice di confusione (soglia 0.5)", ax=axes[0])
RocCurveDisplay.from_predictions(y_test_churn, proba_churn, ax=axes[1], name="logistica")
axes[1].plot([0, 1], [0, 1], "k--", label="caso")
axes[1].set_title("Curva ROC (test)")
PrecisionRecallDisplay.from_predictions(y_test_churn, proba_churn, ax=axes[2], name="logistica")
axes[2].axhline(y_test_churn.mean(), color="k", linestyle="--", label="caso (quota positivi)")
axes[2].set_title("Curva precision-recall (test)")
axes[2].legend()
CalibrationDisplay.from_predictions(y_test_churn, proba_churn, n_bins=8, ax=axes[3], name="logistica")
axes[3].set_title("Calibrazione (test)")
plt.tight_layout()
plt.show()

Con la soglia 0.5 l'accuracy è alta ma il **recall è basso**: il modello segnala solo i clienti con
rischio superiore al 50%, e in un dataset dove il rischio medio è il 7% sono pochissimi. La curva ROC
e l'AUC dicono però che l'ordinamento è buono: i clienti a rischio *ricevono* punteggi più alti. Il
problema è solo dove tagliamo.

### 3.1 Scegliere la soglia in base ai costi

Supponiamo che perdere un cliente senza averlo contattato costi 1.000 € di margine, e che contattare
un cliente che non avrebbe churnato costi 250 € (tempo del customer success più l'offerta di
retention che gli facciamo). Per ogni soglia possiamo calcolare il costo totale atteso, usando le
probabilità stimate in cross-validazione sul training (`cross_val_predict`: ogni account riceve la
probabilità da un modello che non lo ha visto). Scegliamo la soglia che minimizza il costo, e solo
dopo la applichiamo al test.

C'è anche un risultato teorico da verificare: se le probabilità sono calibrate, la soglia ottima è
$c_{FP} / (c_{FP} + c_{FN})$, qui $250 / 1250 = 0.2$. Contattiamo un cliente quando il costo atteso
del non farlo ($p \cdot c_{FN}$) supera quello del farlo ($(1-p) \cdot c_{FP}$).

In [ ]:
COSTO_FN, COSTO_FP = 1000, 250    # euro: cliente perso non contattato / contatto (con offerta) a chi non avrebbe churnato

proba_cv_churn = cross_val_predict(logistica_churn, X_train_churn, y_train_churn, cv=cv, method="predict_proba")[:, 1]
soglie = np.arange(0.02, 0.81, 0.01)
costi = []
for s in soglie:
    vn, fp, fn, vp = confusion_matrix(y_train_churn, (proba_cv_churn >= s).astype(int)).ravel()
    costi.append(COSTO_FN * fn + COSTO_FP * fp)
costi = np.array(costi)
soglia_churn = soglie[costi.argmin()]

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(soglie, costi / 1000)
ax.axvline(soglia_churn, color="red", linestyle="--", label=f"soglia a costo minimo = {soglia_churn:.2f}")
ax.axvline(COSTO_FP / (COSTO_FP + COSTO_FN), color="green", linestyle="-.", label="soglia teorica c_FP/(c_FP+c_FN)")
ax.axvline(0.5, color="gray", linestyle=":", label="soglia di default = 0.5")
ax.set_xlabel("soglia di decisione")
ax.set_ylabel("costo atteso sul training (migliaia di euro)")
ax.set_title("Costo totale in funzione della soglia (probabilità da cross-validazione)")
ax.legend()
plt.show()

pred_churn = (proba_churn >= soglia_churn).astype(int)
m = metriche_binarie(y_test_churn, pred_churn, proba_churn, "Regressione logistica")
m.update(dataset="churn_saas", tipo="binario", soglia=soglia_churn,
         accuracy_ingenua=accuracy_ingenua(y_train_churn, y_test_churn))
risultati.append(m)
print(f"Test con soglia {soglia_churn:.2f}:")
stampa_metriche(m)

def costo_test(pred):
    vn, fp, fn, vp = confusion_matrix(y_test_churn, pred).ravel()
    return COSTO_FN * fn + COSTO_FP * fp
print(f"\nCosto sul test: soglia 0.5 = {costo_test(pred_churn_05):,} €   soglia {soglia_churn:.2f} = {costo_test(pred_churn):,} €"
      f"   nessun modello (nessun contatto) = {COSTO_FN * y_test_churn.sum():,} €")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
mostra_confusione(y_test_churn, pred_churn_05, [0, 1], "Soglia 0.5", ax=axes[0])
mostra_confusione(y_test_churn, pred_churn, [0, 1], f"Soglia {soglia_churn:.2f} (costi)", ax=axes[1])
plt.tight_layout()
plt.show()

Il minimo empirico della curva dei costi cade vicino alla soglia teorica, e la curva è piatta
attorno al minimo: tra 0.15 e 0.35 il costo cambia poco, mentre sale ripidamente sotto 0.1 (contattiamo
tutti) e lentamente verso 0.5 (non contattiamo quasi nessuno). Abbassando la soglia il recall sale e
la precision scende: contattiamo più clienti, alcuni dei quali non avrebbero churnato, ma il costo
totale sul test è inferiore sia a quello della soglia 0.5 sia a quello del "non fare nulla". La
metrica giusta non era né l'accuracy né l'F1: era il costo, e la soglia è un **parametro di
business**, non del modello.

### 3.2 Inferenza con statsmodels

Stimiamo lo stesso modello con `smf.logit` sui dati **non standardizzati**, così i coefficienti sono
"per unità" (per ogni ritardo di pagamento, per ogni ticket, per ogni punto di tasso di utilizzo)
e gli odds ratio si leggono in modo naturale. `C(piano)` e `C(fatturazione)` creano le dummy, con la
prima categoria in ordine alfabetico come riferimento (`Basic` e `annuale`).

In [ ]:
train_churn = X_train_churn.assign(churn=y_train_churn)
test_churn = X_test_churn.assign(churn=y_test_churn)

formula_churn = ("churn ~ tasso_utilizzo + licenze + mesi_anzianita + login_medi_settimana + feature_usate "
                 "+ integrazioni_attive + ticket_90gg + tempo_risposta_medio_ore + ritardi_pagamento_12m "
                 "+ C(piano) + C(fatturazione)")
logit_churn = smf.logit(formula_churn, data=train_churn).fit(disp=0)
print(logit_churn.summary())

**Come leggere il `summary`.** `coef` è la stima sul log-odds, `std err` la sua incertezza,
`P>|z|` il p-value (probabilità di osservare un effetto così grande se il vero coefficiente fosse
zero), `[0.025 0.975]` l'intervallo di confidenza al 95%. `Pseudo R-squ.` (McFadden) non è l'R² della
regressione lineare: valori tra 0.2 e 0.4 indicano già un modello utile.

I coefficienti diventano interpretabili con l'esponenziale: **odds ratio** e relativo intervallo di
confidenza. Un odds ratio di 2 significa "raddoppia le odds di churn"; 0.7 "le riduce del 30%";
1 "nessun effetto" (e se l'intervallo contiene 1, l'effetto non è distinguibile da zero).

In [ ]:
odds = pd.DataFrame({
    "odds_ratio": np.exp(logit_churn.params),
    "IC95_inf": np.exp(logit_churn.conf_int()[0]),
    "IC95_sup": np.exp(logit_churn.conf_int()[1]),
    "p_value": logit_churn.pvalues,
}).drop(index="Intercept").sort_values("odds_ratio", ascending=False)
odds.round(3)

Gli odds ratio sono comodi ma "moltiplicativi"; per rispondere a "di quanti punti percentuali cambia
la probabilità di churn?" usiamo gli **effetti marginali medi** (`get_margeff`): la variazione media
della probabilità per un incremento unitario di ciascuna variabile.

In [ ]:
print(logit_churn.get_margeff().summary())

# Stesso modello, due librerie: le probabilità sul test devono coincidere
proba_sm = logit_churn.predict(test_churn)
print(f"\nROC-AUC sul test: statsmodels = {roc_auc_score(y_test_churn, proba_sm):.4f}   "
      f"scikit-learn = {roc_auc_score(y_test_churn, proba_churn):.4f}")
print(f"Differenza massima tra le probabilità stimate dalle due librerie: {np.abs(proba_sm.values - proba_churn).max():.2e}")

**Cosa portiamo a casa.**

- La regressione logistica produce **probabilità calibrate** e **coefficienti leggibili**: i ritardi
  di pagamento e la fatturazione mensile sono i fattori di rischio più forti, il tasso di utilizzo e
  le integrazioni i più protettivi, con intervalli di confidenza che dicono quanto fidarci.
- La soglia 0.5 non è "il modello": è una convenzione. Con classi sbilanciate la soglia va scelta in
  base ai costi (o al recall che ci serve), su probabilità ottenute in cross-validazione, mai sul test.
- scikit-learn e statsmodels stimano lo stesso modello e danno le stesse probabilità; il primo
  serve a costruire pipeline e valutare, il secondo a spiegare. `class_weight="balanced"` in
  scikit-learn è l'alternativa alla scelta della soglia: riequilibra le classi durante
  l'addestramento, ma rende le probabilità non più calibrate.

## 4. Albero decisionale: `test_flaky`

Un albero di classificazione divide i dati con domande a soglia ("alternanze negli ultimi 20 run ≥
3?"), scegliendo ogni volta la domanda che rende i due gruppi più *puri* (qui con l'indice di Gini).
Ogni foglia predice la classe maggioritaria dei fallimenti che vi finiscono, e la quota di quella
classe è la probabilità stimata. Il risultato è leggibile come una catena di `if/else`, che possiamo
confrontare con l'esperienza di chi lavora sulla CI.

Perché questo dataset: le classi sono quasi bilanciate (accuracy sensata), le evidenze sono regole
(un test che alterna pass e fail mentre nessuno tocca il suo modulo è flaky; un test che fallisce in
sequenza dopo una modifica del modulo, insieme ad altri, è rotto), e non serve alcuno scaling. Il
rischio è il sovra-adattamento: controlliamo profondità e dimensione minima delle foglie in
cross-validazione, come nella regressione.

In [ ]:
feature_flaky = [c for c in flaky.columns if c not in ("id_fallimento", "flaky")]
X_flaky, y_flaky = flaky[feature_flaky], flaky["flaky"]
X_train_flaky, X_test_flaky, y_train_flaky, y_test_flaky = train_test_split(
    X_flaky, y_flaky, test_size=0.2, stratify=y_flaky, random_state=RANDOM_STATE)
print(f"training: {len(X_train_flaky)} fallimenti   test: {len(X_test_flaky)} fallimenti")

griglia_albero = {"max_depth": [1, 2, 3, 4, 5, 6, 8, 10, 15, None],
                  "min_samples_leaf": [5, 10, 20, 40]}      # almeno 5 casi per foglia: una regola basata su un caso solo non è una regola
ricerca_albero = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE), griglia_albero,
                              cv=cv, scoring="accuracy", return_train_score=True, n_jobs=-1)
ricerca_albero.fit(X_train_flaky, y_train_flaky)
print("Iperparametri migliori:", ricerca_albero.best_params_)
print(f"Accuracy in cross-validazione (5 fold): {ricerca_albero.best_score_:.3f}")

ris_albero = pd.DataFrame(ricerca_albero.cv_results_)
foglia_migliore = ricerca_albero.best_params_["min_samples_leaf"]
curva = ris_albero[ris_albero["param_min_samples_leaf"] == foglia_migliore].copy()
curva["profondita"] = curva["param_max_depth"].astype(float).fillna(25).astype(int)
curva = curva.sort_values("profondita")
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(curva["profondita"], curva["mean_train_score"], "o-", label="accuracy training")
ax.plot(curva["profondita"], curva["mean_test_score"], "o-", label="accuracy cross-validazione")
ax.set_xlabel("profondità massima (25 = nessun limite)")
ax.set_ylabel("accuracy")
ax.set_title(f"Albero con min_samples_leaf = {foglia_migliore}: la profondità in più oltre l'ottimo serve solo a memorizzare")
ax.legend()
plt.show()

La curva è piatta oltre pochi livelli: alberi di profondità 3, 5 o 8 hanno accuracy di
cross-validazione quasi identiche, e `GridSearchCV` sceglie il massimo anche quando il vantaggio è
di pochi millesimi, cioè rumore. Applichiamo una **regola di parsimonia**: tra tutte le combinazioni
che perdono meno di un punto percentuale rispetto alla migliore, prendiamo l'albero più semplice
(profondità minima, poi foglie più grandi). È una versione pratica della "regola dell'errore
standard" usata per potare gli alberi, e ci restituisce un modello che il team può leggere per intero.

In [ ]:
tolleranza = 0.01
candidati = ris_albero[ris_albero["mean_test_score"] >= ricerca_albero.best_score_ - tolleranza].copy()
candidati["profondita"] = candidati["param_max_depth"].astype(float).fillna(25)
candidati = candidati.sort_values(["profondita", "param_min_samples_leaf"], ascending=[True, False])
params_semplice = candidati.iloc[0]["params"]
print(f"Combinazioni entro {tolleranza:.0%} dal massimo: {len(candidati)}   →   scelta più semplice: {params_semplice}")
print(f"accuracy in cross-validazione: {candidati.iloc[0]['mean_test_score']:.3f} (massimo: {ricerca_albero.best_score_:.3f})")

albero = DecisionTreeClassifier(random_state=RANDOM_STATE, **params_semplice).fit(X_train_flaky, y_train_flaky)
pred_flaky = albero.predict(X_test_flaky)
proba_flaky = albero.predict_proba(X_test_flaky)[:, 1]
m = metriche_binarie(y_test_flaky, pred_flaky, proba_flaky, "Albero decisionale")
m.update(dataset="test_flaky", tipo="binario", soglia=0.5,
         accuracy_ingenua=accuracy_ingenua(y_train_flaky, y_test_flaky))
risultati.append(m)
print("Test:")
stampa_metriche(m)
print(f"accuracy del classificatore ingenuo (sempre 'rotto'): {m['accuracy_ingenua']:.3f}")

fig, ax = plt.subplots(figsize=(4.8, 4.2))
mostra_confusione(y_test_flaky, pred_flaky, [0, 1], "Matrice di confusione (0 = rotto, 1 = flaky)", ax=ax)
plt.show()

### 4.1 Leggere l'albero

Le regole come testo (`class: 1` = flaky, `class: 0` = rotto), il disegno dei primi livelli e
l'importanza delle feature (quota della riduzione di impurità attribuibile a ciascuna variabile).

In [ ]:
print(f"Profondità dell'albero: {albero.get_depth()}, foglie: {albero.get_n_leaves()}\n")
print(export_text(albero, feature_names=feature_flaky, max_depth=3, decimals=2, show_weights=True))

In [ ]:
fig, ax = plt.subplots(figsize=(24, 9))
plot_tree(albero, feature_names=feature_flaky, class_names=["rotto", "flaky"], max_depth=3,
          filled=True, rounded=True, impurity=False, fontsize=9, ax=ax)
ax.set_title("Albero decisionale (primi 3 livelli; value = [rotti, flaky] nel nodo)")
plt.show()

importanze = pd.Series(albero.feature_importances_, index=feature_flaky).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
importanze.plot.barh(ax=ax, color="steelblue")
ax.set_title("Importanza delle feature")
ax.set_xlabel("quota di riduzione dell'impurità")
plt.show()

**Cosa portiamo a casa.**

- Le domande dell'albero sono quelle che un ingegnere di CI farebbe a mano: il test sta fallendo
  **in sequenza** (più di tre volte di fila)? Allora è rotto, quasi senza eccezioni. Altrimenti, sono
  caduti **altri test nello stesso commit**? Se sì e il test aveva uno storico pulito, è rotto anche
  lui; se falliva già a intermittenza (**alternanze** nello storico, tasso di fallimento non nullo),
  è flaky. La forma "se... e... allora..." è direttamente discutibile con il team.
- Poche regole bastano: la curva di profondità è piatta oltre tre livelli, e la regola di parsimonia
  ci ha restituito otto foglie con un'accuracy quasi identica a quella dell'albero da 46 foglie.
  Il ramo di destra ("più di tre fallimenti consecutivi") è tutto `rotto`: le divisioni successive
  non cambiano la decisione, e una potatura le fonderebbe in una foglia sola.
- `modulo_modificato_nel_commit` ha importanza zero non perché sia inutile, ma perché
  `fallimenti_consecutivi` e `altri_test_falliti` portano già la stessa informazione: con feature
  correlate l'albero ne sceglie una e le altre sembrano irrilevanti. Le importanze misurano quanto
  una variabile è stata *usata*, non l'effetto che avrebbe se la cambiassimo. `durata_media_s`, invece,
  è irrilevante davvero.
- Gli errori residui sono i casi ambigui per costruzione (un test con storico intermittente che
  stavolta è rotto per davvero): nessuna regola li separa, e un albero più profondo li memorizzerebbe
  senza imparare nulla.

## 5. k-Nearest Neighbors: `triage_ticket`

Il kNN classifica un nuovo ticket guardando i **k ticket più simili** già classificati e prendendo
la classe più votata (eventualmente pesando i voti per la vicinanza). Non costruisce regole né
coefficienti: la "conoscenza" è l'archivio stesso. È la logica di chi fa triage per esperienza:
"questo assomiglia ai ticket che di solito girano al team API".

Perché questo dataset: il **multiclasse** viene gratis (i vicini votano, indipendentemente dal
numero di classi), e le classi hanno firme riconoscibili. Due accorgimenti indispensabili, come nella
regressione: **standardizzare** le numeriche (altrimenti `lunghezza_descrizione`, nell'ordine delle
centinaia, domina ogni distanza) e codificare le categoriche in one-hot. L'iperparametro chiave è k:
piccolo = sensibile al rumore, grande = tutto tende alla classe maggioritaria. Lo scegliamo in
cross-validazione con l'**F1 macro**, che dà lo stesso peso alla classe rara `Incidente`.

In [ ]:
cat_triage = ["canale", "tipo_cliente", "componente_indicato", "priorita_dichiarata"]
num_triage = ["stack_trace", "allegati", "lunghezza_descrizione", "parole_errore", "parole_richiesta",
              "punti_interrogativi", "utenti_segnalanti", "ora_apertura"]
X_triage, y_triage = triage[num_triage + cat_triage], triage["categoria"]
X_train_triage, X_test_triage, y_train_triage, y_test_triage = train_test_split(
    X_triage, y_triage, test_size=0.2, stratify=y_triage, random_state=RANDOM_STATE)
print(f"training: {len(X_train_triage)} ticket   test: {len(X_test_triage)} ticket")

prep_triage = ColumnTransformer([
    ("num", StandardScaler(), num_triage),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_triage),
], sparse_threshold=0, verbose_feature_names_out=False)

knn_triage = Pipeline([("prep", prep_triage), ("knn", KNeighborsClassifier())])
griglia_knn = {"knn__n_neighbors": list(range(1, 42, 2)), "knn__weights": ["uniform", "distance"]}
ricerca_knn = GridSearchCV(knn_triage, griglia_knn, cv=cv, scoring="f1_macro", n_jobs=-1)
ricerca_knn.fit(X_train_triage, y_train_triage)
print("Iperparametri migliori:", ricerca_knn.best_params_)
print(f"F1 macro in cross-validazione (5 fold): {ricerca_knn.best_score_:.3f}")

ris_knn = pd.DataFrame(ricerca_knn.cv_results_)
fig, ax = plt.subplots(figsize=(9, 4.2))
for pesi in ["uniform", "distance"]:
    sotto = ris_knn[ris_knn["param_knn__weights"] == pesi]
    ax.plot(sotto["param_knn__n_neighbors"].astype(int), sotto["mean_test_score"], "o-", markersize=3,
            label=f"weights = {pesi}")
ax.set_xlabel("k (numero di vicini)")
ax.set_ylabel("F1 macro in cross-validazione")
ax.set_title("Scelta di k")
ax.legend()
plt.show()

In [ ]:
classi = list(ricerca_knn.best_estimator_.classes_)
pred_triage = ricerca_knn.predict(X_test_triage)
proba_triage = ricerca_knn.predict_proba(X_test_triage)
m = metriche_multiclasse(y_test_triage, pred_triage, proba_triage, classi, "k-Nearest Neighbors")
m.update(dataset="triage_ticket", tipo="multiclasse (4)", soglia=np.nan,
         accuracy_ingenua=accuracy_ingenua(y_train_triage, y_test_triage))
risultati.append(m)
print("Test (precision, recall e F1 sono medie macro):")
stampa_metriche(m)
print(f"accuracy del classificatore ingenuo (sempre 'Bug'): {m['accuracy_ingenua']:.3f}\n")
print(classification_report(y_test_triage, pred_triage, digits=3))

ordine = ["Bug", "Incidente", "Richiesta", "Domanda"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
mostra_confusione(y_test_triage, pred_triage, ordine, "Conteggi", ax=axes[0])
mostra_confusione(y_test_triage, pred_triage, ordine, "Normalizzata per classe vera (recall sulla diagonale)",
                  ax=axes[1], normalizza="true")
plt.tight_layout()
plt.show()

### 5.1 Guardare i vicini

Come nella regressione, ogni previsione del kNN si spiega mostrando i vicini. Prendiamo un ticket
del test, recuperiamo i 5 ticket del training più simili e vediamo come hanno "votato".

In [ ]:
pipe_knn = ricerca_knn.best_estimator_
prep_addestrato, knn_addestrato = pipe_knn.named_steps["prep"], pipe_knn.named_steps["knn"]

esempio = X_test_triage.iloc[[3]]
distanze, indici = knn_addestrato.kneighbors(prep_addestrato.transform(esempio), n_neighbors=5)
print("Ticket da classificare (dal test set):")
print(esempio.T.to_string(header=False))
print(f"\nCategoria vera: {y_test_triage.iloc[3]}   |   prevista dal kNN: {ricerca_knn.predict(esempio)[0]}")
print("Probabilità per classe:", dict(zip(classi, ricerca_knn.predict_proba(esempio)[0].round(2).tolist())))
print("\nI 5 ticket del training più simili e la loro categoria:")
X_train_triage.iloc[indici[0]].assign(categoria=y_train_triage.iloc[indici[0]].values,
                                      distanza=distanze[0].round(2))

**Cosa portiamo a casa.**

- Il kNN supera nettamente il classificatore ingenuo senza che gli abbiamo descritto nulla delle
  classi: la "firma" di ogni categoria è contenuta nell'archivio dei ticket già classificati.
- La matrice di confusione normalizzata racconta dove sbaglia: la coppia più confusa è
  **Richiesta / Domanda** (stesso tono, stessi canali, nessuna parola di errore), seguita da qualche
  bug preso per richiesta. Gli **incidenti**, la classe rara, vengono riconosciuti quasi tutti: il
  numero di utenti che segnalano e la priorità dichiarata li isolano bene. Se per il business l'errore
  grave fosse un incidente classificato come bug, potremmo abbassare la soglia sulla classe `Incidente`
  usando le probabilità, a costo di più falsi allarmi.
- Le curve di k mostrano che con un solo vicino il modello è instabile, con troppi appiattisce le
  classi rare verso `Bug`: l'F1 macro lo segnala prima dell'accuracy, perché la classe che soffre per
  prima è proprio `Incidente`. La pesatura per distanza aiuta, perché lascia contare di più i ticket
  davvero simili anche quando k è generoso.
- Ogni previsione richiede di scandire l'intero archivio e di conservarlo: su 2.000 ticket è
  istantaneo, su milioni servono strutture ad hoc o un altro modello.

## 6. Rete neurale MLP: `defect_pr`

Un percettrone multistrato applica alle feature una sequenza di trasformazioni non lineari (strati
di neuroni con attivazione ReLU) e chiude con un neurone logistico che produce la probabilità della
classe positiva. Rispetto alla regressione logistica, la rete può rappresentare **interazioni**
(rischio alto solo quando due condizioni valgono insieme) e **soglie** senza che nessuno le
specifichi a mano. Il prezzo è un numero di parametri molto più alto, quindi la tendenza al
sovra-adattamento, e l'assenza di coefficienti da leggere.

Perché questo dataset: l'analisi esplorativa ha mostrato che il rischio di bug nasce da
combinazioni (modifica grande **senza** test, modulo poco coperto **e** complesso, merge in orari
infelici) che un modello additivo può solo approssimare. Il confronto con una regressione logistica
sugli stessi dati ci dirà quanto vale, in AUC, la capacità di rappresentare le interazioni.

Tre scelte da motivare:

- **regolarizzazione L2 (`alpha`)** scelta in cross-validazione: con 4.000 PR e il 12% di positivi,
  una rete poco regolarizzata memorizza il rumore e fa *peggio* del modello lineare. Vedremo la
  curva.
- **niente early stopping**: in `MLPClassifier` si basa sull'accuracy di una validazione interna,
  che con classi sbilanciate è piatta e quasi inutile come segnale; preferiamo un numero massimo di
  epoche e la regolarizzazione.
- **soglia scelta sulla curva precision-recall** ottenuta in cross-validazione, come per il churn:
  `MLPClassifier` non ha `class_weight`, ma non ne abbiamo bisogno se lavoriamo sulle probabilità.

In [ ]:
num_log_defect = ["righe_aggiunte", "righe_rimosse", "file_modificati"]          # code lunghe: log1p prima dello scaling
num_defect = ["moduli_toccati", "complessita_media", "test_aggiunti", "copertura_modulo", "esperienza_autore_mesi",
              "commit_autore_modulo", "reviewer", "commenti_review", "ora_merge"]
cat_defect = ["tipo", "giorno_merge"]
X_defect, y_defect = defect[num_log_defect + num_defect + cat_defect], defect["bug_entro_30gg"]
X_train_defect, X_test_defect, y_train_defect, y_test_defect = train_test_split(
    X_defect, y_defect, test_size=0.2, stratify=y_defect, random_state=RANDOM_STATE)
print(f"training: {len(X_train_defect)} PR ({y_train_defect.mean():.1%} con bug)   "
      f"test: {len(X_test_defect)} PR ({y_test_defect.mean():.1%} con bug)")

prep_defect = ColumnTransformer([
    ("num_log", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                          ("scala", StandardScaler())]), num_log_defect),
    ("num", StandardScaler(), num_defect),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_defect),
], sparse_threshold=0, verbose_feature_names_out=False)

logistica_defect = Pipeline([("prep", prep_defect), ("logit", LogisticRegression(max_iter=5000))])
mlp_defect = Pipeline([("prep", prep_defect),
                       ("mlp", MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", solver="adam",
                                             max_iter=1500, random_state=RANDOM_STATE))])

griglia_mlp = {"mlp__alpha": [0.01, 0.1, 1.0, 10.0]}
ricerca_mlp = GridSearchCV(mlp_defect, griglia_mlp, cv=cv, scoring="roc_auc", return_train_score=True, n_jobs=-1)
ricerca_mlp.fit(X_train_defect, y_train_defect)
ris_mlp = pd.DataFrame(ricerca_mlp.cv_results_)[["param_mlp__alpha", "mean_train_score", "mean_test_score", "std_test_score"]]
ris_mlp.columns = ["alpha", "AUC training", "AUC cross-validazione", "dev. std."]
print("Effetto della regolarizzazione (ROC-AUC):")
print(ris_mlp.round(3).to_string(index=False))
print("\nalpha migliore:", ricerca_mlp.best_params_["mlp__alpha"])

La tabella dice tutto: con `alpha` piccolo l'AUC di training è altissimo e quello di cross-validazione
basso (sovra-adattamento); con `alpha` grande la rete si semplifica troppo. Fissato l'`alpha`
migliore, confrontiamo la rete con la regressione logistica sugli stessi dati e con le stesse
pieghe, usando due metriche indipendenti dalla soglia: ROC-AUC e average precision (l'area sotto la
curva precision-recall, più severa quando i positivi sono rari).

In [ ]:
mlp_migliore = ricerca_mlp.best_estimator_
for nome, modello in [("Regressione logistica (confronto)", logistica_defect), ("Rete neurale MLP", mlp_migliore)]:
    auc = cross_val_score(modello, X_train_defect, y_train_defect, cv=cv, scoring="roc_auc", n_jobs=-1)
    ap = cross_val_score(modello, X_train_defect, y_train_defect, cv=cv, scoring="average_precision", n_jobs=-1)
    print(f"{nome:34s} ROC-AUC = {auc.mean():.3f} ± {auc.std():.3f}  (per piega: {np.round(auc, 3)})"
          f"   average precision = {ap.mean():.3f}")

### 6.1 Addestramento finale, curva di apprendimento e soglia

Addestriamo la rete su tutto il training e guardiamo la **loss** per epoca (l'entropia incrociata
che l'ottimizzatore minimizza). Poi scegliamo la soglia: otteniamo le probabilità in cross-validazione
sul training, tracciamo precision e recall per ogni soglia e scegliamo quella che massimizza l'F1.
In un contesto reale il criterio potrebbe essere diverso, per esempio "vogliamo rileggere al massimo
il 20% delle PR" oppure "vogliamo intercettare almeno il 70% dei bug": il meccanismo è lo stesso.

In [ ]:
mlp_migliore.fit(X_train_defect, y_train_defect)
rete = mlp_migliore.named_steps["mlp"]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(rete.loss_curve_)
ax.set_title(f"Loss di training per epoca (entropia incrociata), {rete.n_iter_} epoche")
ax.set_xlabel("epoca")
ax.set_ylabel("loss")
plt.show()

proba_cv_defect = cross_val_predict(mlp_migliore, X_train_defect, y_train_defect, cv=cv, method="predict_proba")[:, 1]
soglie = np.arange(0.05, 0.71, 0.01)
tabella_soglie = pd.DataFrame({
    "soglia": soglie,
    "precision": [precision_score(y_train_defect, proba_cv_defect >= s, zero_division=0) for s in soglie],
    "recall": [recall_score(y_train_defect, proba_cv_defect >= s) for s in soglie],
    "F1": [f1_score(y_train_defect, proba_cv_defect >= s) for s in soglie],
    "quota_PR_segnalate": [(proba_cv_defect >= s).mean() for s in soglie],
})
soglia_defect = tabella_soglie.loc[tabella_soglie["F1"].idxmax(), "soglia"]

fig, ax = plt.subplots(figsize=(9, 4.2))
for col in ["precision", "recall", "F1", "quota_PR_segnalate"]:
    ax.plot(tabella_soglie["soglia"], tabella_soglie[col], label=col)
ax.axvline(soglia_defect, color="red", linestyle="--", label=f"soglia scelta = {soglia_defect:.2f}")
ax.set_xlabel("soglia di decisione")
ax.set_title("Precision, recall e F1 in funzione della soglia (probabilità da cross-validazione)")
ax.legend()
plt.show()
print(tabella_soglie[tabella_soglie["soglia"].round(2).isin([0.1, 0.2, 0.3, 0.4, 0.5])].round(3).to_string(index=False))

In [ ]:
logistica_defect.fit(X_train_defect, y_train_defect)
proba_defect_mlp = mlp_migliore.predict_proba(X_test_defect)[:, 1]
proba_defect_lin = logistica_defect.predict_proba(X_test_defect)[:, 1]

pred_defect = (proba_defect_mlp >= soglia_defect).astype(int)
m = metriche_binarie(y_test_defect, pred_defect, proba_defect_mlp, "Rete neurale MLP")
m.update(dataset="defect_pr", tipo="binario", soglia=soglia_defect,
         accuracy_ingenua=accuracy_ingenua(y_train_defect, y_test_defect))
risultati.append(m)
m_lin = metriche_binarie(y_test_defect, (proba_defect_lin >= soglia_defect).astype(int), proba_defect_lin,
                         "Regressione logistica (confronto)")
print(f"Test, soglia {soglia_defect:.2f}")
print("Rete neurale MLP:                  ", end=""); stampa_metriche(m)
print("Regressione logistica (confronto): ", end=""); stampa_metriche(m_lin)
print(f"average precision sul test: MLP = {average_precision_score(y_test_defect, proba_defect_mlp):.3f}   "
      f"logistica = {average_precision_score(y_test_defect, proba_defect_lin):.3f}")
print(f"accuracy del classificatore ingenuo (sempre 'nessun bug'): {m['accuracy_ingenua']:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
mostra_confusione(y_test_defect, pred_defect, [0, 1], f"MLP, soglia {soglia_defect:.2f}", ax=axes[0])
RocCurveDisplay.from_predictions(y_test_defect, proba_defect_mlp, ax=axes[1], name="MLP")
RocCurveDisplay.from_predictions(y_test_defect, proba_defect_lin, ax=axes[1], name="logistica")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("Curve ROC (test)")
PrecisionRecallDisplay.from_predictions(y_test_defect, proba_defect_mlp, ax=axes[2], name="MLP")
PrecisionRecallDisplay.from_predictions(y_test_defect, proba_defect_lin, ax=axes[2], name="logistica")
axes[2].set_title("Curve precision-recall (test)")
plt.tight_layout()
plt.show()

Per capire *dove* la rete guadagna, confrontiamo i due modelli sulle combinazioni viste nell'analisi
esplorativa: per ogni gruppo di PR del test (grande/piccola × con/senza test) mettiamo a fianco il
tasso di bug reale e la probabilità media stimata da ciascun modello.

In [ ]:
gruppi = pd.DataFrame({
    "modifica": np.where(X_test_defect["righe_aggiunte"] + X_test_defect["righe_rimosse"] > 300, "> 300 righe", "≤ 300 righe"),
    "test_aggiunti": X_test_defect["test_aggiunti"].values,
    "tasso_bug_reale": y_test_defect.values,
    "stima_MLP": proba_defect_mlp,
    "stima_logistica": proba_defect_lin,
})
gruppi.groupby(["modifica", "test_aggiunti"]).agg(n=("tasso_bug_reale", "size"), tasso_bug_reale=("tasso_bug_reale", "mean"),
                                                   stima_MLP=("stima_MLP", "mean"), stima_logistica=("stima_logistica", "mean")).round(3)

**Cosa portiamo a casa.**

- A parità di dati e preparazione la rete ottiene un ordinamento migliore (AUC e average precision
  più alti in cross-validazione e sul test) perché rappresenta le congiunzioni: nel gruppo "modifica
  grande senza test" la sua stima media segue il tasso di bug reale, mentre il modello logistico,
  costretto a sommare gli effetti, lo sottostima nettamente; nei gruppi a basso rischio le due stime
  quasi coincidono. Il guadagno è di pochi punti di AUC, ma alla soglia scelta si traduce in molti
  più bug intercettati a parità di PR da rileggere.
- Il guadagno esiste solo con la **regolarizzazione giusta**: la stessa architettura con `alpha`
  piccolo fa peggio della regressione logistica. Con pochi dati e classi sbilanciate, la
  regolarizzazione è il primo iperparametro da cercare, prima della dimensione degli strati.
- La soglia, di nuovo, è una decisione separata dal modello: a 0.5 la rete intercetterebbe pochi
  bug; la soglia scelta sulla curva precision-recall in cross-validazione bilancia PR da rileggere
  e bug intercettati. Il criterio va concordato con chi farà le review.
- Il prezzo è l'opacità: la rete non dice *perché* una PR è rischiosa. Se al team serve spiegarlo,
  una regressione logistica con le interazioni scritte a mano (le stesse scoperte nell'EDA) o un
  insieme di alberi con le importanze sono alternative da valutare.

## 7. Riepilogo

Le metriche di test dei quattro modelli. Per il triage, precision, recall e F1 sono medie macro e
la ROC-AUC è "uno contro tutti"; per churn e PR la soglia è quella scelta in cross-validazione, non
0.5. La colonna `accuracy_ingenua` è l'accuracy del classificatore che predice sempre la classe più
frequente: il confronto con `accuracy` mostra perché, con classi sbilanciate, l'accuracy non basta.

In [ ]:
tabella = pd.DataFrame(risultati)[["modello", "dataset", "tipo", "soglia", "accuracy", "accuracy_ingenua",
                                   "precision", "recall", "F1", "ROC_AUC"]]
tabella.round(3)

**Le lezioni trasversali.**

1. **Il bilanciamento delle classi decide tutto il resto.** Stratificazione, metriche, soglia:
   ogni scelta discende dalla quota di positivi vista nell'analisi esplorativa.
2. **La matrice di confusione prima delle metriche.** Ogni numero (precision, recall, F1) è una
   lettura parziale di quella tabella; i due tipi di errore hanno quasi sempre costi diversi.
3. **La soglia è una decisione, non un default.** Il modello produce probabilità; il taglio si
   sceglie in base a costi o vincoli operativi, su probabilità ottenute in cross-validazione, e si
   verifica una sola volta sul test.
4. **ROC-AUC e average precision confrontano i modelli, precision e recall descrivono la decisione.**
   Le prime non dipendono dalla soglia, le seconde sì: servono entrambe.
5. **Ogni algoritmo ha il suo terreno.** La regressione logistica quando servono probabilità
   calibrate e coefficienti spiegabili (e statsmodels per l'inferenza); l'albero quando il problema è
   fatto di regole; il kNN quando "casi simili" è la domanda giusta e le classi sono più di due; la rete
   quando il rischio nasce da combinazioni, purché la regolarizziamo e abbiamo abbastanza dati.

**Esercizi possibili.** Applicare ogni algoritmo a un dataset diverso e spiegare cosa cambia;
aggiungere a mano alla regressione logistica sulle PR le interazioni scoperte nell'EDA e vedere
quanto del divario con la rete si recupera; provare `class_weight="balanced"` sul churn e confrontare
la calibrazione; sostituire l'albero con una random forest sui test flaky; cambiare i costi
FN/FP del churn e osservare come si sposta la soglia.